In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# load files
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')

# ensure timestamp numeric
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce')
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce')

# incident window in UTC+8: May 26, 2020 05:00 to 05:30
start_dt = tz.localize(datetime(2020, 5, 26, 5, 0, 0))
end_dt = tz.localize(datetime(2020, 5, 26, 5, 30, 0))
start_epoch = start_dt.timestamp()
end_epoch = end_dt.timestamp()

# helper to format epoch -> ISO in UTC+8
def fmt_epoch_to_asia_shanghai(ep):
    if pd.isna(ep):
        return None
    return datetime.fromtimestamp(float(ep), tz).isoformat()

# file-level summaries
metric_rows = len(metric_df)
trace_rows = len(trace_df)

metric_ts_min = metric_df['timestamp'].min() if metric_rows>0 else np.nan
metric_ts_max = metric_df['timestamp'].max() if metric_rows>0 else np.nan
trace_ts_min = trace_df['timestamp'].min() if trace_rows>0 else np.nan
trace_ts_max = trace_df['timestamp'].max() if trace_rows>0 else np.nan

metric_rows_in_window = int(metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].shape[0])
trace_rows_in_window = int(trace_df[(trace_df['timestamp'] >= start_epoch) & (trace_df['timestamp'] <= end_epoch)].shape[0])

file_summary_df = pd.DataFrame([
    {
        'file': 'metric.csv',
        'rows': metric_rows,
        'ts_min': fmt_epoch_to_asia_shanghai(metric_ts_min),
        'ts_max': fmt_epoch_to_asia_shanghai(metric_ts_max),
        'rows_in_window': metric_rows_in_window
    },
    {
        'file': 'trace.csv',
        'rows': trace_rows,
        'ts_min': fmt_epoch_to_asia_shanghai(trace_ts_min),
        'ts_max': fmt_epoch_to_asia_shanghai(trace_ts_max),
        'rows_in_window': trace_rows_in_window
    }
]).set_index('file')

# Grouped stats for metric: group by cmdb_id and kpi_name
metric_groups_full = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        ts_min=('timestamp', 'min'),
        ts_max=('timestamp', 'max'),
        P95=('value', lambda s: np.nan if s.dropna().shape[0]==0 else np.nanpercentile(s.dropna(), 95)),
        P5 =('value', lambda s: np.nan if s.dropna().shape[0]==0 else np.nanpercentile(s.dropna(), 5))
    )
    .reset_index()
)

# convert ts fields to readable Asia/Shanghai ISO for display, but keep numeric ts in full table
metric_groups_full['ts_min_iso'] = metric_groups_full['ts_min'].apply(fmt_epoch_to_asia_shanghai)
metric_groups_full['ts_max_iso'] = metric_groups_full['ts_max'].apply(fmt_epoch_to_asia_shanghai)

# top 20 by count
metric_top20 = (
    metric_groups_full
    .sort_values('count', ascending=False)
    .loc[:, ['cmdb_id','kpi_name','count','ts_min_iso','ts_max_iso','P95','P5']]
    .rename(columns={'ts_min_iso':'ts_min','ts_max_iso':'ts_max'})
    .head(20)
)

# Grouped stats for trace: group by cmdb_id and trace_name
trace_groups_full = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        ts_min=('timestamp', 'min'),
        ts_max=('timestamp', 'max'),
        P95=('value', lambda s: np.nan if s.dropna().shape[0]==0 else np.nanpercentile(s.dropna(), 95)),
        P5 =('value', lambda s: np.nan if s.dropna().shape[0]==0 else np.nanpercentile(s.dropna(), 5))
    )
    .reset_index()
)

trace_groups_full['ts_min_iso'] = trace_groups_full['ts_min'].apply(fmt_epoch_to_asia_shanghai)
trace_groups_full['ts_max_iso'] = trace_groups_full['ts_max'].apply(fmt_epoch_to_asia_shanghai)

trace_top20 = (
    trace_groups_full
    .sort_values('count', ascending=False)
    .loc[:, ['cmdb_id','trace_name','count','ts_min_iso','ts_max_iso','P95','P5']]
    .rename(columns={'ts_min_iso':'ts_min','ts_max_iso':'ts_max'})
    .head(20)
)

# Display compact outputs (file-level summary, metric top20, trace top20)
file_summary_df, metric_top20, trace_top20

```
Out[1]:
```
```python
# Compose a plain-English summary based on previously computed variables:
# file_summary_df, metric_top20, trace_top20 are available in the kernel.

metric_file = file_summary_df.loc['metric.csv']
trace_file = file_summary_df.loc['trace.csv']

# prepare top-5 snippets for metric and trace
metric_top5 = metric_top20.head(5).to_dict('records')
trace_top5 = trace_top20.head(5).to_dict('records')

summary_lines = []
summary_lines.append("File-level summary:")
summary_lines.append(f"- metric.csv: {int(metric_file['rows'])} rows; timestamps {metric_file['ts_min']} to {metric_file['ts_max']}; {int(metric_file['rows_in_window'])} rows inside 2020-05-26 05:00-05:30 (UTC+8).")
summary_lines.append(f"- trace.csv: {int(trace_file['rows'])} rows; timestamps {trace_file['ts_min']} to {trace_file['ts_max']}; {int(trace_file['rows_in_window'])} rows inside 2020-05-26 05:00-05:30 (UTC+8).")
summary_lines.append("")
summary_lines.append("Top metric component-KPI groups (top 5 by sample count):")
for r in metric_top5:
    summary_lines.append(f"- {r['cmdb_id']} | {r['kpi_name']} : count={int(r['count'])}, ts_min={r['ts_min']}, ts_max={r['ts_max']}, P95={r['P95']}, P5={r['P5']}")

summary_lines.append("")
summary_lines.append("Top trace component-trace groups (top 5 by sample count):")
for r in trace_top5:
    summary_lines.append(f"- {r['cmdb_id']} | {r['trace_name']} : count={int(r['count'])}, ts_min={r['ts_min']}, ts_max={r['ts_max']}, P95={r['P95']}, P5={r['P5']}")

summary_lines.append("")
summary_lines.append("Observations:")
summary_lines.append("- Many metric groups have 61 samples (likely covering 61 minutes from 04:30 to 05:30 inclusive).")
summary_lines.append("- All top trace groups have 30 samples (covering 05:00 to 05:29 inclusive).")
summary_lines.append("- The metric file contains much more data overall (91,191 rows) while the trace file is smaller (5,520 rows).")
summary_lines.append("- The incident window (05:00–05:30 UTC+8) contains ~45,648 metric rows and all 5,520 trace rows.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             rows                     ts_min                     ts_max  rows_in_window
file                                                                                   
metric.csv  91191  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00           45648
trace.csv    5520  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00            5520,       cmdb_id                  kpi_name  count                     ts_min                     ts_max         P95         P5
1772   os_022  Outgoing_network_traffic     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    0.080220   0.006849
1204   os_011      Memory_available_pct     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00   59.530000  59.510000
1267   os_012      Processor_load_5_min     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    0.016667   0.001667
987    os_007               Disk_rd_ios     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    0.000000   0.000000
327    db_008              CPU_free_pct     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00   98.821525  97.476677
97     db_003              CPU_free_pct     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00   99.284716  97.186037
1406   os_015  Incoming_network_traffic     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    0.003477   0.000000
1135   os_010              CPU_util_pct     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    0.136145   0.092617
1068   os_008                Send_total     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    0.000679   0.000512
5      db_001              CPU_free_pct     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00   99.394409  98.087094
1649   os_020              Disk_io_util     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    0.000000   0.000000
1792  osb_001              app.avg_time     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    0.022498   0.000533
1793  osb_001                   app.num     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00  619.000000  30.000000
1084   os_009              CPU_util_pct     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    0.536387   0.215394
727    os_002              CPU_util_pct     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    2.730599   1.608821
1527   os_017                Send_total     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    0.452203   0.026333
966    os_006                Send_total     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    0.000470   0.000306
1241   os_012              Disk_io_util     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    2.000000   0.000000
1794  osb_001            app.succee_num     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00  619.000000  30.000000
1795  osb_001           app.succee_rate     61  2020-05-26T04:30:00+08:00  2020-05-26T05:30:00+08:00    1.000000   1.000000,        cmdb_id                                     trace_name  count                     ts_min                     ts_max          P95          P5
0   docker_001  trace.from_os_022.remoteprocess.duration_mean     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.523040    0.256598
1   docker_001   trace.from_os_022.remoteprocess.duration_p95     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     1.710355    0.628833
2   docker_001     trace.from_os_022.remoteprocess.error_rate     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.000000    0.000000
3   docker_001      trace.from_os_022.remoteprocess.row_count     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00   162.650000   32.150000
4   docker_001                   trace.self.csf.duration_mean     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.054058    0.033250
5   docker_001                    trace.self.csf.duration_p95     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.174807    0.065163
6   docker_001                      trace.self.csf.error_rate     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.000000    0.000000
7   docker_001                       trace.self.csf.row_count     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00   650.600000  128.600000
8   docker_001             trace.self.flyremote.duration_mean     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.010832    0.004954
9   docker_001              trace.self.flyremote.duration_p95     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.061040    0.006000
10  docker_001                trace.self.flyremote.error_rate     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.000000    0.000000
11  docker_001                 trace.self.flyremote.row_count     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00   162.650000   32.150000
12  docker_001                  trace.self.jdbc.duration_mean     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.003117    0.001832
13  docker_001                   trace.self.jdbc.duration_p95     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.003000    0.003000
14  docker_001                     trace.self.jdbc.error_rate     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.000000    0.000000
15  docker_001                      trace.self.jdbc.row_count     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00  2603.850000  515.300000
16  docker_001                 trace.self.local.duration_mean     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.012246    0.004300
17  docker_001                  trace.self.local.duration_p95     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.089000    0.007698
18  docker_001                    trace.self.local.error_rate     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00     0.000000    0.000000
19  docker_001                     trace.self.local.row_count     30  2020-05-26T05:00:00+08:00  2020-05-26T05:29:00+08:00  2277.100000  450.100000)```
```

In [ ]:
# In[2]:

# Reuse existing variables from kernel:
# metric_df, metric_groups_full, start_epoch, end_epoch, tz, fmt_epoch_to_asia_shanghai

import numpy as np
import pandas as pd

# 1) Filter metric rows within incident window (inclusive)
metric_df_window = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()

# 2) Attach global thresholds (P95, P5) to window rows so we can evaluate highs/lows
thresholds = metric_groups_full[['cmdb_id','kpi_name','P95','P5']].copy()
metric_win_with_thresh = metric_df_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# 3) Compute per-row flags using the global thresholds (do NOT recompute thresholds here)
metric_win_with_thresh['is_high'] = (metric_win_with_thresh['value'] >= metric_win_with_thresh['P95']).astype(int)
metric_win_with_thresh['is_low']  = (metric_win_with_thresh['value'] <= metric_win_with_thresh['P5']).astype(int)

# 4) Aggregate within-window stats per group (only groups that have at least one sample in window)
agg_funcs = {
    'value': ['count','max','min'],
    'is_high': 'sum',
    'is_low': 'sum'
}
temp = metric_win_with_thresh.groupby(['cmdb_id','kpi_name'], dropna=False).agg(agg_funcs)
# flatten columns
temp.columns = ['sample_count_in_window','max_value_in_window','min_value_in_window','count_high','count_low']
temp = temp.reset_index()

# 5) Earliest timestamps where high/low occurred
high_times = metric_win_with_thresh[metric_win_with_thresh['is_high'] == 1].groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_high_ts'})
low_times  = metric_win_with_thresh[metric_win_with_thresh['is_low']  == 1].groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_low_ts'})

# 6) Merge aggregated counts and earliest times with the global groups to include groups with zero samples in window
metric_anomaly_full = metric_groups_full[['cmdb_id','kpi_name','P95','P5']].merge(
    temp, on=['cmdb_id','kpi_name'], how='left'
).merge(
    high_times, on=['cmdb_id','kpi_name'], how='left'
).merge(
    low_times, on=['cmdb_id','kpi_name'], how='left'
)

# 7) Fill missing numeric counts with zeros and keep max/min as NaN when no samples
metric_anomaly_full['sample_count_in_window'] = metric_anomaly_full['sample_count_in_window'].fillna(0).astype(int)
metric_anomaly_full['count_high'] = metric_anomaly_full['count_high'].fillna(0).astype(int)
metric_anomaly_full['count_low']  = metric_anomaly_full['count_low'].fillna(0).astype(int)

# 8) Convert earliest timestamps to ISO (Asia/Shanghai) or None
metric_anomaly_full['earliest_high_ts_iso'] = metric_anomaly_full['earliest_high_ts'].apply(lambda x: fmt_epoch_to_asia_shanghai(x) if not pd.isna(x) else None)
metric_anomaly_full['earliest_low_ts_iso']  = metric_anomaly_full['earliest_low_ts'].apply(lambda x: fmt_epoch_to_asia_shanghai(x) if not pd.isna(x) else None)

# 9) Keep max/min values as-is (NaN if no samples). For display, round numeric P95/P5 and max/min for compactness
metric_anomaly_full['P95'] = metric_anomaly_full['P95'].astype(float)
metric_anomaly_full['P5']  = metric_anomaly_full['P5'].astype(float)

# 10) Severity metric and sorting
metric_anomaly_full['max_count'] = metric_anomaly_full[['count_high','count_low']].max(axis=1)
metric_anomaly_full = metric_anomaly_full.sort_values(['max_count','sample_count_in_window'], ascending=[False,False])

# 11) Prepare compact top-20 table with requested columns
metric_anomaly_top20 = metric_anomaly_full.loc[:, [
    'cmdb_id','kpi_name','sample_count_in_window','count_high','count_low',
    'earliest_high_ts_iso','earliest_low_ts_iso','max_value_in_window','min_value_in_window','P95','P5'
]].head(20).copy()

# 12) For compactness, round numeric display columns to reasonable precision
for col in ['max_value_in_window','min_value_in_window','P95','P5']:
    if col in metric_anomaly_top20.columns:
        metric_anomaly_top20[col] = metric_anomaly_top20[col].map(lambda v: round(v,6) if pd.notna(v) else v)

# 13) Store full result in kernel variable metric_anomaly_full (already set) and display top-20 summary + shape of full table
metric_anomaly_top20, metric_anomaly_full.shape

```
Out[2]:
```
```python
# Compose a plain-English summary of the metric-window scan results and display it.
# Reuse kernel variables: metric_anomaly_full, metric_anomaly_top20

num_groups_total = metric_anomaly_full.shape[0]
num_groups_with_samples = int((metric_anomaly_full['sample_count_in_window'] > 0).sum())
total_metric_rows_in_window = int(metric_anomaly_full['sample_count_in_window'].sum())

# Prepare a few representative examples from top-20 (already computed)
examples = metric_anomaly_top20.head(10).to_dict('records')

summary_lines = []
summary_lines.append(f"Summary of metric anomalies within 2020-05-26 05:00–05:30 (UTC+8):")
summary_lines.append(f"- Total component-KPI groups considered: {num_groups_total}.")
summary_lines.append(f"- Groups with >=1 sample in the window: {num_groups_with_samples}.")
summary_lines.append(f"- Total metric samples in the window (all groups): {total_metric_rows_in_window}.")
summary_lines.append("")
summary_lines.append("Top findings (from the top-20 groups by anomaly count):")
summary_lines.append("- Many groups have every sample in-window flagged as both 'high' and 'low' relative to their global P95/P5.")
summary_lines.append("  This occurs when the group's P95 and P5 equal the actual constant value for that metric (i.e., the series is constant),")
summary_lines.append("  so count_high == count_low == sample_count_in_window. Examples:")
for r in examples[:10]:
    summary_lines.append(f"  - {r['cmdb_id']} | {r['kpi_name']}: samples={r['sample_count_in_window']}, "
                         f"count_high={r['count_high']}, count_low={r['count_low']}, "
                         f"P95={r['P95']}, P5={r['P5']}, earliest_high={r['earliest_high_ts_iso']}, earliest_low={r['earliest_low_ts_iso']}")
summary_lines.append("")
summary_lines.append("Notable repeated patterns in top groups:")
summary_lines.append("- Several OS metrics (e.g., Disk_rd_ios, Disk_io_util) are constant-zero across the window.")
summary_lines.append("- Some service/db metrics (e.g., Hang, MEM_Total, On_Off_State, Row_Lock) are constant across the window (same P95 and P5).")
summary_lines.append("- Earliest high/low timestamps for these groups are 2020-05-26T05:00:00+08:00 (the window start) in the top examples.")
summary_lines.append("")
summary_lines.append("Operational note:")
summary_lines.append("- Because many groups are constant and their global P95/P5 equal the observed values, the raw counts of 'high'/'low' are not by themselves "
                     "definitive indicators of transient spikes. They can reflect stable metrics equal to their thresholds. Interpret these results alongside "
                     "absolute max/min values and domain knowledge.")
summary_lines.append("")
summary_lines.append("Data availability:")
summary_lines.append("- Full per-group anomaly table is stored in the kernel variable: metric_anomaly_full")
summary_lines.append("- Top-20 summary table is stored in: metric_anomaly_top20")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id         kpi_name  sample_count_in_window  count_high  count_low       earliest_high_ts_iso        earliest_low_ts_iso  max_value_in_window  min_value_in_window         P95          P5
987    os_007      Disk_rd_ios                      31          31         31  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             0.000000             0.000000    0.000000    0.000000
1649   os_020     Disk_io_util                      31          31         31  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             0.000000             0.000000    0.000000    0.000000
1795  osb_001  app.succee_rate                      31          31         31  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             1.000000             1.000000    1.000000    1.000000
10     db_001             Hang                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             1.000000             1.000000    1.000000    1.000000
15     db_001        MEM_Total                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00           381.902264           381.902264  381.902264  381.902264
21     db_001     On_Off_State                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             1.000000             1.000000    1.000000    1.000000
28     db_001         Row_Lock                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             0.000000             0.000000    0.000000    0.000000
30     db_001  SctRead_Per_Sec                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             0.000000             0.000000    0.000000    0.000000
47     db_002             AIOS                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             0.000000             0.000000    0.000000    0.000000
56     db_002             Hang                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             1.000000             1.000000    1.000000    1.000000
61     db_002        MEM_Total                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00           381.902264           381.902264  381.902264  381.902264
67     db_002     On_Off_State                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             1.000000             1.000000    1.000000    1.000000
74     db_002         Row_Lock                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             0.000000             0.000000    0.000000    0.000000
76     db_002  SctRead_Per_Sec                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             0.000000             0.000000    0.000000    0.000000
102    db_003             Hang                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             1.000000             1.000000    1.000000    1.000000
107    db_003        MEM_Total                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00           381.902264           381.902264  381.902264  381.902264
113    db_003     On_Off_State                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             1.000000             1.000000    1.000000    1.000000
120    db_003         Row_Lock                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             0.000000             0.000000    0.000000    0.000000
148    db_004             Hang                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00             1.000000             1.000000    1.000000    1.000000
153    db_004        MEM_Total                      30          30         30  2020-05-26T05:00:00+08:00  2020-05-26T05:00:00+08:00           381.902264           381.902264  381.902264  381.902264, (2060, 14))```
```

In [ ]:
# In[3]:

# Reuse existing variables: metric_df, metric_df_window, metric_groups_full, metric_anomaly_full, tz, fmt_epoch_to_asia_shanghai
import pandas as pd
import numpy as np

# Ensure window df exists; if not, recreate
try:
    metric_df_window
except NameError:
    metric_df_window = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()

# Attach global thresholds (P95, P5) from metric_groups_full
thresholds = metric_groups_full[['cmdb_id','kpi_name','P95','P5']].copy()
win = metric_df_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Determine breach type per row: 'high' if value>=P95, 'low' if value<=P5, 'both' if both, else 'none'
def determine_breach(row):
    ge = pd.notna(row['P95']) and pd.notna(row['value']) and (row['value'] >= row['P95'])
    le = pd.notna(row['P5'])  and pd.notna(row['value']) and (row['value'] <= row['P5'])
    if ge and le:
        return 'both'
    if ge:
        return 'high'
    if le:
        return 'low'
    return 'none'

win = win.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)
win['breach'] = win.apply(determine_breach, axis=1)

# Create run ids: increment when breach changes or when group key changes. Only consecutive identical breach values form a run.
# We want runs only for breach != 'none'; but we still compute runs across all to identify segments.
win['grp_key'] = win['cmdb_id'].astype(str) + '||' + win['kpi_name'].astype(str)
# detect boundary where grp_key or breach changes
win['prev_key'] = win['grp_key'].shift(1)
win['prev_breach'] = win['breach'].shift(1)
win['new_run'] = (win['grp_key'] != win['prev_key']) | (win['breach'] != win['prev_breach'])
# first row is new run
win.loc[0, 'new_run'] = True
# run id as cumulative sum of new_run flags within DataFrame
win['run_id'] = win['new_run'].cumsum()

# Aggregate runs, but only keep runs where breach != 'none'
run_groups = win[win['breach'] != 'none'].groupby(['grp_key','run_id','cmdb_id','kpi_name','breach'], as_index=False)

segments = run_groups.agg(
    start_ts=('timestamp','min'),
    end_ts=('timestamp','max'),
    length=('timestamp','count'),
    max_value_in_segment=('value','max'),
    min_value_in_segment=('value','min'),
    P95=('P95','first'),
    P5=('P5','first')
)

# Convert timestamps and compute severities
def compute_severity(row):
    p95 = row['P95']
    p5 = row['P5']
    maxv = row['max_value_in_segment']
    minv = row['min_value_in_segment']
    # avoid zero-division
    s_high = (maxv - p95) / max(1e-9, p95) if pd.notna(p95) else np.nan
    s_low  = (p5 - minv) / max(1e-9, p5) if pd.notna(p5) else np.nan
    if row['breach'] == 'high':
        return s_high
    if row['breach'] == 'low':
        return s_low
    # both: take the larger relative deviation
    return np.nanmax([s_high, s_low])

segments['severity'] = segments.apply(compute_severity, axis=1)
segments['start_ts_iso'] = segments['start_ts'].apply(lambda x: fmt_epoch_to_asia_shanghai(x))
segments['end_ts_iso']   = segments['end_ts'].apply(lambda x: fmt_epoch_to_asia_shanghai(x))

# Store full detected segments table in kernel variable as requested
metric_fault_segments_full = segments.copy()

# Now aggregate per component-KPI considering only segments with length >= 3
segments_ge3 = segments[segments['length'] >= 3].copy()

# If there are no segments >=3, ensure downstream code handles empty
if not segments_ge3.empty:
    agg = segments_ge3.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        num_segments=('run_id','nunique'),
        longest_segment_length=('length','max'),
        earliest_segment_start_ts=('start_ts','min'),
        max_segment_severity=('severity','max')
    )
    # Determine breach_type_of_longest: get the breach type of the segment with longest length (tie -> earliest start)
    # Merge to find breach_type_of_longest
    # First find for each group the run(s) with max length
    segs_for_longest = segments_ge3.merge(
        agg[['cmdb_id','kpi_name','longest_segment_length']],
        on=['cmdb_id','kpi_name'],
        how='inner'
    )
    # Filter to rows where length == longest_segment_length, then pick earliest start
    segs_for_longest = segs_for_longest[segs_for_longest['length'] == segs_for_longest['longest_segment_length']]
    segs_for_longest_sorted = segs_for_longest.sort_values(['cmdb_id','kpi_name','start_ts'])
    first_longest = segs_for_longest_sorted.groupby(['cmdb_id','kpi_name'], as_index=False).first()[['cmdb_id','kpi_name','breach','start_ts']]
    first_longest = first_longest.rename(columns={'breach':'breach_type_of_longest','start_ts':'earliest_longest_start_ts'})
    agg = agg.merge(first_longest[['cmdb_id','kpi_name','breach_type_of_longest','earliest_longest_start_ts']], on=['cmdb_id','kpi_name'], how='left')
    # Convert earliest timestamps to ISO
    agg['earliest_segment_start_ts_iso'] = agg['earliest_segment_start_ts'].apply(lambda x: fmt_epoch_to_asia_shanghai(x) if pd.notna(x) else None)
    agg['earliest_longest_start_ts_iso'] = agg['earliest_longest_start_ts'].apply(lambda x: fmt_epoch_to_asia_shanghai(x) if pd.notna(x) else None)
else:
    # Empty aggregation
    agg = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','num_segments','longest_segment_length',
        'earliest_segment_start_ts','max_segment_severity','breach_type_of_longest','earliest_longest_start_ts'
    ])

# Merge window-level max/min and global P95/P5 (these are in metric_anomaly_full)
summary = agg.merge(
    metric_anomaly_full[['cmdb_id','kpi_name','max_value_in_window','min_value_in_window','P95','P5']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# For rows where earliest_longest_start_ts_iso not present, use earliest_segment_start_ts_iso
if 'earliest_longest_start_ts_iso' in summary.columns:
    summary['earliest_segment_start_ts_iso'] = summary['earliest_longest_start_ts_iso'].fillna(summary.get('earliest_segment_start_ts_iso'))
else:
    summary['earliest_segment_start_ts_iso'] = summary.get('earliest_segment_start_ts_iso')

# Keep required columns and format
final_cols = [
    'cmdb_id','kpi_name','num_segments','longest_segment_length','earliest_segment_start_ts_iso',
    'breach_type_of_longest','max_value_in_window','min_value_in_window','P95','P5','max_segment_severity'
]
# Ensure max_segment_severity column exists (it was named max_segment_severity in agg)
if 'max_segment_severity' in summary.columns:
    summary = summary.rename(columns={'max_segment_severity':'max_segment_severity'})
else:
    # fallback rename if different
    if 'max_segment_severity' not in summary.columns and 'max_segment_severity' not in summary.columns:
        # maybe named max_segment_severity already as max_segment_severity; create if missing
        summary['max_segment_severity'] = summary.get('max_segment_severity', np.nan)

# Ensure all expected columns exist
for c in final_cols:
    if c not in summary.columns:
        summary[c] = None

# Round numeric columns for compactness
for c in ['max_value_in_window','min_value_in_window','P95','P5','max_segment_severity']:
    if c in summary.columns:
        summary[c] = summary[c].apply(lambda v: round(v,6) if pd.notna(v) else v)

# Sort by longest_segment_length desc, then max_segment_severity desc
summary_sorted = summary.sort_values(['longest_segment_length','max_segment_severity'], ascending=[False, False])

# Top 20 summary
metric_fault_summary_top20 = summary_sorted.loc[:, final_cols].head(20).reset_index(drop=True)

# Output: keep full segments table and top-20 summary (display top-20 as required)
metric_fault_segments_full.shape, metric_fault_summary_top20

```
Out[3]:
```
```python
# Summarize detected breach segments using existing kernel variables:
# metric_fault_segments_full, metric_anomaly_full, metric_fault_summary_top20

import pandas as pd
import numpy as np

segs = metric_fault_segments_full.copy()

total_segments = int(segs.shape[0])
segs_ge3 = segs[segs['length'] >= 3].copy()
groups_with_segs_ge3 = int(segs_ge3.groupby(['cmdb_id','kpi_name']).ngroups)
total_groups = int(metric_anomaly_full.shape[0]) if 'metric_anomaly_full' in globals() else None

# how many segments span the full 05:00–05:30 window (31 points) or 30 points
full_window_len = 31
count_full_window_segments = int(segs_ge3[segs_ge3['length'] >= full_window_len].shape[0])
# severity zero segments (likely where value == P95 == P5)
count_zero_severity = int((segs_ge3['severity'] == 0).sum())

# Most common breach type among segments length>=3
breach_counts = segs_ge3['breach'].value_counts().to_dict()

# Prepare a brief list of top examples (from the earlier top-20 summary if present)
examples = []
if 'metric_fault_summary_top20' in globals():
    ex_df = metric_fault_summary_top20.head(5)
    for _, r in ex_df.iterrows():
        examples.append(f"{r['cmdb_id']}|{r['kpi_name']} (longest={r['longest_segment_length']}, type={r['breach_type_of_longest']}, start={r['earliest_segment_start_ts_iso']})")

summary_lines = [
    "Breach-segment scan summary (window: 2020-05-26 05:00–05:30 UTC+8):",
    f"- Total detected breach segments in window: {total_segments}.",
    f"- Segments with length >=3: {int(segs_ge3.shape[0])}, across {groups_with_segs_ge3} unique component-KPI groups (out of {total_groups} groups).",
    f"- Segments that span the full window (>= {full_window_len} points): {count_full_window_segments}.",
    f"- Segments with zero computed severity (severity==0): {count_zero_severity}.",
    f"- Breach type distribution among length>=3 segments: {breach_counts}.",
    "",
    "Key findings:",
    "- Many component-KPI series produce a single long consecutive breach segment covering the entire incident window (length 30–31).",
    "- A large portion of these long segments are labeled 'both' (value >= P95 and <= P5) and have zero severity — this indicates the series is constant and equals its global thresholds (P95 == P5 == value).",
    "- Examples from the top results: " + ("; ".join(examples) if examples else "(no examples available)"),
    "",
    "Operational note:",
    "- Long segments of breach that are 'both' with zero severity are likely benign artifacts of constant-valued series relative to their global thresholds; "
    "they should be interpreted carefully rather than treated as high-severity anomalies.",
    "",
    "Data objects stored in kernel:",
    "- Full detected segments: metric_fault_segments_full",
    "- Top-group summary (top 20): metric_fault_summary_top20",
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((5431, 15),     cmdb_id         kpi_name  num_segments  longest_segment_length earliest_segment_start_ts_iso breach_type_of_longest  max_value_in_window  min_value_in_window         P95          P5  max_segment_severity
0    os_007      Disk_rd_ios             1                      31     2020-05-26T05:00:00+08:00                   both             0.000000             0.000000    0.000000    0.000000                   0.0
1    os_020     Disk_io_util             1                      31     2020-05-26T05:00:00+08:00                   both             0.000000             0.000000    0.000000    0.000000                   0.0
2   osb_001  app.succee_rate             1                      31     2020-05-26T05:00:00+08:00                   both             1.000000             1.000000    1.000000    1.000000                   0.0
3    db_001             Hang             1                      30     2020-05-26T05:00:00+08:00                   both             1.000000             1.000000    1.000000    1.000000                   0.0
4    db_001        MEM_Total             1                      30     2020-05-26T05:00:00+08:00                   both           381.902264           381.902264  381.902264  381.902264                   0.0
5    db_001     On_Off_State             1                      30     2020-05-26T05:00:00+08:00                   both             1.000000             1.000000    1.000000    1.000000                   0.0
6    db_001         Row_Lock             1                      30     2020-05-26T05:00:00+08:00                   both             0.000000             0.000000    0.000000    0.000000                   0.0
7    db_001  SctRead_Per_Sec             1                      30     2020-05-26T05:00:00+08:00                   both             0.000000             0.000000    0.000000    0.000000                   0.0
8    db_002             AIOS             1                      30     2020-05-26T05:00:00+08:00                   both             0.000000             0.000000    0.000000    0.000000                   0.0
9    db_002             Hang             1                      30     2020-05-26T05:00:00+08:00                   both             1.000000             1.000000    1.000000    1.000000                   0.0
10   db_002        MEM_Total             1                      30     2020-05-26T05:00:00+08:00                   both           381.902264           381.902264  381.902264  381.902264                   0.0
11   db_002     On_Off_State             1                      30     2020-05-26T05:00:00+08:00                   both             1.000000             1.000000    1.000000    1.000000                   0.0
12   db_002         Row_Lock             1                      30     2020-05-26T05:00:00+08:00                   both             0.000000             0.000000    0.000000    0.000000                   0.0
13   db_002  SctRead_Per_Sec             1                      30     2020-05-26T05:00:00+08:00                   both             0.000000             0.000000    0.000000    0.000000                   0.0
14   db_003             Hang             1                      30     2020-05-26T05:00:00+08:00                   both             1.000000             1.000000    1.000000    1.000000                   0.0
15   db_003        MEM_Total             1                      30     2020-05-26T05:00:00+08:00                   both           381.902264           381.902264  381.902264  381.902264                   0.0
16   db_003     On_Off_State             1                      30     2020-05-26T05:00:00+08:00                   both             1.000000             1.000000    1.000000    1.000000                   0.0
17   db_003         Row_Lock             1                      30     2020-05-26T05:00:00+08:00                   both             0.000000             0.000000    0.000000    0.000000                   0.0
18   db_004             Hang             1                      30     2020-05-26T05:00:00+08:00                   both             1.000000             1.000000    1.000000    1.000000                   0.0
19   db_004        MEM_Total             1                      30     2020-05-26T05:00:00+08:00                   both           381.902264           381.902264  381.902264  381.902264                   0.0)```
```

In [ ]:
# In[4]:

# Filter stored segments for heavy faults and produce requested summaries.
# Reuse kernel variables: metric_fault_segments_full, tz, fmt_epoch_to_asia_shanghai

import pandas as pd
import numpy as np

segs = metric_fault_segments_full.copy()

# Ensure numeric severity exists
segs['severity'] = pd.to_numeric(segs['severity'], errors='coerce')

# Primary threshold: 0.5 (50%)
th = 0.5
candidate = segs[(segs['length'] >= 3) & (segs['severity'] > th)].copy()

# If none match, relax to 0.2 (20%)
if candidate.shape[0] == 0:
    th = 0.2
    candidate = segs[(segs['length'] >= 3) & (segs['severity'] > th)].copy()

# Store filtered segments in kernel variable
metric_heavy_faults = candidate.copy()

# Prepare display columns and formatting
if not candidate.empty:
    # Ensure ISO timestamp columns exist; create if absent
    if 'start_ts_iso' not in candidate.columns:
        candidate['start_ts_iso'] = candidate['start_ts'].apply(lambda x: fmt_epoch_to_asia_shanghai(x) if pd.notna(x) else None)
    if 'end_ts_iso' not in candidate.columns:
        candidate['end_ts_iso'] = candidate['end_ts'].apply(lambda x: fmt_epoch_to_asia_shanghai(x) if pd.notna(x) else None)
    display_cols = [
        'cmdb_id','kpi_name','breach','start_ts_iso','end_ts_iso','length',
        'max_value_in_segment','min_value_in_segment','P95','P5','severity'
    ]
    metric_heavy_top20 = (
        candidate
        .sort_values('severity', ascending=False)
        .loc[:, display_cols]
        .head(20)
        .copy()
    )
    # Round numeric columns for compactness
    for c in ['max_value_in_segment','min_value_in_segment','P95','P5','severity']:
        if c in metric_heavy_top20.columns:
            metric_heavy_top20[c] = metric_heavy_top20[c].map(lambda v: round(v,6) if pd.notna(v) else v)

    # Small summary
    number_of_segments_matching = int(candidate.shape[0])
    number_of_unique_components_matching = int(candidate['cmdb_id'].nunique())
    # earliest start ts
    earliest_ts_val = candidate['start_ts'].min()
    earliest_segment_start_ts_iso = fmt_epoch_to_asia_shanghai(earliest_ts_val) if pd.notna(earliest_ts_val) else None
    # up to 10 distinct cmdb_id candidates sorted by number of segments descending
    comp_counts = candidate.groupby('cmdb_id').size().reset_index(name='segment_count').sort_values('segment_count', ascending=False)
    top_components = comp_counts.head(10).reset_index(drop=True)
    # Prepare summary DataFrame compact
    heavy_summary_df = pd.DataFrame({
        'number_of_segments_matching':[number_of_segments_matching],
        'number_of_unique_components_matching':[number_of_unique_components_matching],
        'earliest_segment_start_ts_iso':[earliest_segment_start_ts_iso],
        'severity_threshold_used':[th]
    })
    # also keep top_components separately for clarity
    metric_heavy_top20, heavy_summary_df, top_components
else:
    # No segments even after relaxation
    metric_heavy_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','breach','start_ts_iso','end_ts_iso','length',
        'max_value_in_segment','min_value_in_segment','P95','P5','severity'
    ])
    heavy_summary_df = pd.DataFrame({
        'number_of_segments_matching':[0],
        'number_of_unique_components_matching':[0],
        'earliest_segment_start_ts_iso':[None],
        'severity_threshold_used':[th]
    })
    top_components = pd.DataFrame(columns=['cmdb_id','segment_count'])
    metric_heavy_top20, heavy_summary_df, top_components

```
Out[4]:
```
```python
# Compose a plain-English summary based on kernel variables from previous steps.
# Uses: metric_heavy_top20, heavy_summary_df, top_components, metric_heavy_faults (if present)
import pandas as pd

# Safely fetch summary info
if 'heavy_summary_df' in globals() and not heavy_summary_df.empty:
    row = heavy_summary_df.iloc[0]
    num_segments = int(row['number_of_segments_matching'])
    num_components = int(row['number_of_unique_components_matching'])
    earliest_ts = row['earliest_segment_start_ts_iso']
    severity_threshold_used = float(row['severity_threshold_used'])
else:
    # fallback if variable missing
    if 'metric_heavy_faults' in globals():
        df = metric_heavy_faults
        num_segments = int(df.shape[0])
        num_components = int(df['cmdb_id'].nunique()) if num_segments>0 else 0
        earliest_ts = df['start_ts'].min()
        earliest_ts = None if pd.isna(earliest_ts) else earliest_ts
        severity_threshold_used = 0.5
    else:
        num_segments = 0
        num_components = 0
        earliest_ts = None
        severity_threshold_used = 0.5

# Top components list
top_list = []
if 'top_components' in globals() and not top_components.empty:
    for _, r in top_components.iterrows():
        top_list.append(f"{r['cmdb_id']} (segments={int(r['segment_count'])})")
else:
    # try derive from metric_heavy_faults
    if 'metric_heavy_faults' in globals() and not metric_heavy_faults.empty:
        comp_counts = metric_heavy_faults.groupby('cmdb_id').size().reset_index(name='segment_count').sort_values('segment_count', ascending=False).head(10)
        for _, r in comp_counts.iterrows():
            top_list.append(f"{r['cmdb_id']} (segments={int(r['segment_count'])})")

# Build summary text
if num_segments == 0:
    summary = (
        f"No breach segments with length>=3 exceeded the severity threshold.\n"
        f"Checked thresholds: attempted 50% then relaxed to 20% if needed; "
        f"final threshold used = {severity_threshold_used:.2f}.\n"
        "No heavy faults found in metric_fault_segments_full for the incident window."
    )
else:
    top_components_str = ", ".join(top_list) if top_list else "(none)"
    summary = (
        f"Heavy breach-segment summary (threshold used: {severity_threshold_used:.2f} severity):\n"
        f"- Number of matching segments: {num_segments}.\n"
        f"- Number of unique components with matching segments: {num_components}.\n"
        f"- Earliest matching segment start: {earliest_ts}.\n"
        f"- Up to 10 top components by segment count: {top_components_str}.\n\n"
        f"Notes:\n"
        f"- The full filtered segments are stored in kernel variable: metric_heavy_faults\n"
        f"- The top-20 segments table (ordered by severity) is in: metric_heavy_top20\n"
        f"- Inspect those objects for per-segment details (timestamps, lengths, P95/P5, and severity values)."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[5]:

# Scan trace.csv in the incident window for consecutive breach segments using global P95/P5 thresholds.
# Reuse kernel variables: trace_df, trace_groups_full, start_epoch, end_epoch, tz, fmt_epoch_to_asia_shanghai

import pandas as pd
import numpy as np

# 1. Prepare windowed trace data
try:
    trace_df_window
except NameError:
    trace_df_window = trace_df[(trace_df['timestamp'] >= start_epoch) & (trace_df['timestamp'] <= end_epoch)].copy()

# 2. Ensure thresholds (P95/P5) per (cmdb_id, trace_name) are available
# trace_groups_full was computed earlier; if not, compute it now from full trace_df
if 'trace_groups_full' not in globals():
    trace_groups_full = (
        trace_df
        .groupby(['cmdb_id','trace_name'], dropna=False)
        .agg(
            count=('value','count'),
            ts_min=('timestamp','min'),
            ts_max=('timestamp','max'),
            P95=('value', lambda s: np.nan if s.dropna().shape[0]==0 else np.nanpercentile(s.dropna(),95)),
            P5 =('value', lambda s: np.nan if s.dropna().shape[0]==0 else np.nanpercentile(s.dropna(),5))
        )
        .reset_index()
    )

thresholds = trace_groups_full[['cmdb_id','trace_name','P95','P5']].copy()

# 3. Merge thresholds into windowed data
win = trace_df_window.merge(thresholds, on=['cmdb_id','trace_name'], how='left')

# 4. Determine breach type per row
def determine_breach_trace(row):
    ge = pd.notna(row['P95']) and pd.notna(row['value']) and (row['value'] >= row['P95'])
    le = pd.notna(row['P5'])  and pd.notna(row['value']) and (row['value'] <= row['P5'])
    if ge and le:
        return 'both'
    if ge:
        return 'high'
    if le:
        return 'low'
    return 'none'

win = win.sort_values(['cmdb_id','trace_name','timestamp']).reset_index(drop=True)
win['breach'] = win.apply(determine_breach_trace, axis=1)

# 5. Build run ids for consecutive identical breaches within each trace series
win['grp_key'] = win['cmdb_id'].astype(str) + '||' + win['trace_name'].astype(str)
win['prev_key'] = win['grp_key'].shift(1)
win['prev_breach'] = win['breach'].shift(1)
win['new_run'] = (win['grp_key'] != win['prev_key']) | (win['breach'] != win['prev_breach'])
win.loc[0, 'new_run'] = True
win['run_id'] = win['new_run'].cumsum()

# 6. Aggregate runs where breach != 'none'
run_rows = win[win['breach'] != 'none'].copy()
run_groups = run_rows.groupby(['grp_key','run_id','cmdb_id','trace_name','breach'], as_index=False)

segments = run_groups.agg(
    start_ts=('timestamp','min'),
    end_ts=('timestamp','max'),
    length=('timestamp','count'),
    max_value_in_segment=('value','max'),
    min_value_in_segment=('value','min'),
    P95=('P95','first'),
    P5 =('P5','first')
)

# 7. Compute severity per the definition, avoiding division by zero
def compute_sev_trace(row):
    p95 = row['P95']
    p5 = row['P5']
    maxv = row['max_value_in_segment']
    minv = row['min_value_in_segment']
    if row['breach'] == 'high':
        return (maxv - p95) / max(1e-9, p95) if pd.notna(p95) else np.nan
    if row['breach'] == 'low':
        return (p5 - minv) / max(1e-9, p5) if pd.notna(p5) else np.nan
    # both: take larger relative deviation (if both present)
    s_high = (maxv - p95) / max(1e-9, p95) if pd.notna(p95) else np.nan
    s_low  = (p5 - minv) / max(1e-9, p5) if pd.notna(p5) else np.nan
    return np.nanmax([s_high, s_low])

segments['severity'] = segments.apply(compute_sev_trace, axis=1)
segments['start_ts_iso'] = segments['start_ts'].apply(lambda x: fmt_epoch_to_asia_shanghai(x))
segments['end_ts_iso']   = segments['end_ts'].apply(lambda x: fmt_epoch_to_asia_shanghai(x))

# 8. Store full segments table in kernel variable
trace_fault_segments_full = segments.copy()

# 9. Filter by severity threshold: primary 0.5, fallback 0.2
th = 0.5
candidates = segments[segments['severity'] > th].copy()
if candidates.shape[0] == 0:
    th = 0.2
    candidates = segments[segments['severity'] > th].copy()

# 10. Prepare top-20 table ordered by severity desc and compact formatting
if not candidates.empty:
    display_cols = [
        'cmdb_id','trace_name','breach','start_ts_iso','end_ts_iso','length',
        'max_value_in_segment','min_value_in_segment','P95','P5','severity'
    ]
    trace_fault_top20 = candidates.sort_values('severity', ascending=False).loc[:, display_cols].head(20).copy()
    for c in ['max_value_in_segment','min_value_in_segment','P95','P5','severity']:
        trace_fault_top20[c] = trace_fault_top20[c].map(lambda v: round(v,6) if pd.notna(v) else v)

    # 11. Small summary
    number_of_segments_matching = int(candidates.shape[0])
    number_of_unique_components_matching = int(candidates['cmdb_id'].nunique())
    earliest_ts_val = candidates['start_ts'].min()
    earliest_segment_start_ts_iso = fmt_epoch_to_asia_shanghai(earliest_ts_val) if pd.notna(earliest_ts_val) else None

    comp_counts = candidates.groupby('cmdb_id').size().reset_index(name='segment_count').sort_values('segment_count', ascending=False)
    top_components = comp_counts.head(10).reset_index(drop=True)

    small_summary = pd.DataFrame({
        'number_of_segments_matching':[number_of_segments_matching],
        'number_of_unique_components_matching':[number_of_unique_components_matching],
        'earliest_segment_start_ts_iso':[earliest_segment_start_ts_iso],
        'severity_threshold_used':[th]
    })
else:
    # No matches even after relaxation
    trace_fault_top20 = pd.DataFrame(columns=[
        'cmdb_id','trace_name','breach','start_ts_iso','end_ts_iso','length',
        'max_value_in_segment','min_value_in_segment','P95','P5','severity'
    ])
    small_summary = pd.DataFrame({
        'number_of_segments_matching':[0],
        'number_of_unique_components_matching':[0],
        'earliest_segment_start_ts_iso':[None],
        'severity_threshold_used':[th]
    })
    top_components = pd.DataFrame(columns=['cmdb_id','segment_count'])

# 12. Return compact outputs: top-20 table and small summary + top components
trace_fault_top20, small_summary, top_components

```
Out[5]:
```
```python
# Plain-English summary of trace fault scan results (from kernel variables)
summary = (
    "Trace-segment scan (window 2020-05-26 05:00–05:30 UTC+8):\n"
    "- Severity threshold used: 0.50 (no relaxation needed).\n"
    "- Matching segments: 8 segments across 4 unique components.\n"
    "- Earliest matching segment start: 2020-05-26T05:16:00+08:00.\n"
    "- Top candidate components (by segment count): docker_003 (5), docker_002 (1), docker_004 (1), os_021 (1).\n\n"
    "Notable high-severity segments (top examples):\n"
    "1) docker_003 | trace.self.jdbc.duration_p95 — high, 2020-05-26T05:22:00+08:00, length=1, max=0.050500, P95=0.008202, severity≈5.16\n"
    "2) docker_004 | trace.self.local.duration_p95 — low, 2020-05-26T05:16–05:17+08:00, length=2, min=0.008000, P5=0.061705, severity≈0.87\n"
    "3) docker_003 | trace.self.local.duration_p95 — low, 2020-05-26T05:17–05:18+08:00, length=2, min=0.008000, P5=0.040400, severity≈0.80\n"
    "4) docker_002 | trace.self.local.duration_p95 — high, 2020-05-26T05:19–05:20+08:00, length=2, max=2.083300, P95=1.217482, severity≈0.71\n"
    "5) os_021 / docker_003 remoteprocess.duration_p95 (paired) — low, 2020-05-26T05:17–05:18+08:00, length=2, severity≈0.62\n\n"
    "Observations & guidance:\n"
    "- docker_003 is the primary hotspot (5 of 8 segments), including the single largest severity event (jdbc.duration_p95).\n"
    "- Most detected segments are short (1–2 minutes); investigate the context around 05:16–05:22 for transient trace spikes/drops.\n"
    "- Also inspect interactions between docker_003 and os_021 (paired remoteprocess trace segments at ~05:17).\n\n"
    "Data location in kernel:\n"
    "- Full detected trace segments: trace_fault_segments_full\n"
    "- Top-20 segments table shown earlier: trace_fault_top20\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                      trace_name breach               start_ts_iso                 end_ts_iso  length  max_value_in_segment  min_value_in_segment       P95        P5  severity
212  docker_003                    trace.self.jdbc.duration_p95   high  2020-05-26T05:22:00+08:00  2020-05-26T05:22:00+08:00       1              0.050500              0.050500  0.008202  0.003000  5.156660
289  docker_004                   trace.self.local.duration_p95    low  2020-05-26T05:16:00+08:00  2020-05-26T05:17:00+08:00       2              0.060850              0.008000  0.095842  0.061705  0.870351
224  docker_003                   trace.self.local.duration_p95    low  2020-05-26T05:17:00+08:00  2020-05-26T05:18:00+08:00       2              0.008000              0.008000  0.098000  0.040400  0.801980
140  docker_002                   trace.self.local.duration_p95   high  2020-05-26T05:19:00+08:00  2020-05-26T05:20:00+08:00       2              2.083300              1.463700  1.217482  0.080800  0.711154
488      os_021  trace.to_docker_003.remoteprocess.duration_p95    low  2020-05-26T05:17:00+08:00  2020-05-26T05:18:00+08:00       2              0.627000              0.322150  2.519413  0.838613  0.615854
178  docker_003    trace.from_os_021.remoteprocess.duration_p95    low  2020-05-26T05:17:00+08:00  2020-05-26T05:18:00+08:00       2              0.627000              0.322150  2.519413  0.838613  0.615854
185  docker_003                    trace.self.csf.duration_mean   high  2020-05-26T05:21:00+08:00  2020-05-26T05:22:00+08:00       2              0.115947              0.078515  0.076275  0.034834  0.520111
220  docker_003                  trace.self.local.duration_mean    low  2020-05-26T05:17:00+08:00  2020-05-26T05:18:00+08:00       2              0.003684              0.003033  0.021527  0.006277  0.516834,    number_of_segments_matching  number_of_unique_components_matching earliest_segment_start_ts_iso  severity_threshold_used
0                            8                                     4     2020-05-26T05:16:00+08:00                      0.5,       cmdb_id  segment_count
0  docker_003              5
1  docker_002              1
2  docker_004              1
3      os_021              1)```
```

In [ ]:
# In[6]:

# Filter trace_fault_segments_full for cmdb_id == 'docker_003' and produce requested outputs.
# Reuse existing kernel variables: trace_fault_segments_full, fmt_epoch_to_asia_shanghai

import pandas as pd
import numpy as np

# Ensure the source variable exists
try:
    trace_fault_segments_full
except NameError:
    raise NameError("trace_fault_segments_full not found in kernel. Run the trace scan step first.")

# Filter for docker_003
docker_003_trace_faults = trace_fault_segments_full[trace_fault_segments_full['cmdb_id'] == 'docker_003'].copy()

# Prepare display columns
display_cols = [
    'cmdb_id','trace_name','breach','start_ts_iso','end_ts_iso','length',
    'max_value_in_segment','min_value_in_segment','P95','P5','severity'
]

if docker_003_trace_faults.shape[0] == 0:
    # Empty results: prepare empty top table and summary
    docker_003_trace_faults_top20 = pd.DataFrame(columns=display_cols)
    docker_003_summary = pd.DataFrame([{
        'earliest_start_ts_iso': None,
        'most_severe_trace_name': None,
        'most_severe_start_ts_iso': None,
        'most_severe_breach': None,
        'most_severe_severity': None,
        'num_segments': 0,
        'num_unique_trace_names': 0
    }])
else:
    # Ensure ISO timestamp columns exist
    if 'start_ts_iso' not in docker_003_trace_faults.columns:
        docker_003_trace_faults['start_ts_iso'] = docker_003_trace_faults['start_ts'].apply(lambda x: fmt_epoch_to_asia_shanghai(x) if pd.notna(x) else None)
    if 'end_ts_iso' not in docker_003_trace_faults.columns:
        docker_003_trace_faults['end_ts_iso'] = docker_003_trace_faults['end_ts'].apply(lambda x: fmt_epoch_to_asia_shanghai(x) if pd.notna(x) else None)

    # Top-20 by severity descending
    docker_003_trace_faults_sorted = docker_003_trace_faults.sort_values('severity', ascending=False)
    docker_003_trace_faults_top20 = docker_003_trace_faults_sorted.loc[:, display_cols].head(20).copy()

    # Round numeric columns for compactness
    for c in ['max_value_in_segment','min_value_in_segment','P95','P5','severity']:
        if c in docker_003_trace_faults_top20.columns:
            docker_003_trace_faults_top20[c] = docker_003_trace_faults_top20[c].map(lambda v: round(v,6) if pd.notna(v) else v)

    # Earliest start timestamp among docker_003 segments
    # Prefer the numeric start_ts if available for accurate min
    if 'start_ts' in docker_003_trace_faults.columns:
        earliest_ts_val = docker_003_trace_faults['start_ts'].min()
        earliest_start_ts_iso = fmt_epoch_to_asia_shanghai(earliest_ts_val) if pd.notna(earliest_ts_val) else None
    else:
        # fallback to existing ISO column min lexicographically (should be fine for ISO)
        earliest_start_ts_iso = docker_003_trace_faults['start_ts_iso'].min()

    # Most severe segment row (handle all-NaN severity safely)
    if docker_003_trace_faults['severity'].dropna().shape[0] == 0:
        most_severe_row = None
        most_severe_trace_name = None
        most_severe_start_ts_iso = None
        most_severe_breach = None
        most_severe_severity = None
    else:
        idx = docker_003_trace_faults['severity'].idxmax()
        row = docker_003_trace_faults.loc[idx]
        most_severe_trace_name = row.get('trace_name')
        most_severe_start_ts_iso = row.get('start_ts_iso') if pd.notna(row.get('start_ts_iso')) else (fmt_epoch_to_asia_shanghai(row.get('start_ts')) if pd.notna(row.get('start_ts')) else None)
        most_severe_breach = row.get('breach')
        most_severe_severity = round(row.get('severity'),6) if pd.notna(row.get('severity')) else None

    docker_003_summary = pd.DataFrame([{
        'earliest_start_ts_iso': earliest_start_ts_iso,
        'most_severe_trace_name': most_severe_trace_name,
        'most_severe_start_ts_iso': most_severe_start_ts_iso,
        'most_severe_breach': most_severe_breach,
        'most_severe_severity': most_severe_severity,
        'num_segments': int(docker_003_trace_faults.shape[0]),
        'num_unique_trace_names': int(docker_003_trace_faults['trace_name'].nunique())
    }])

# Return compact outputs: top-20 table and the 1-row summary
docker_003_trace_faults_top20, docker_003_summary

```
Out[6]:
```
```python
# Plain-English summary for docker_003 trace fault segments (based on trace_fault_segments_full filter)
summary = (
    "docker_003 trace-segment summary (window 2020-05-26 05:00–05:30 UTC+8):\n"
    "- Total detected segments for docker_003: 66 segments across 24 distinct trace names.\n"
    "- Earliest segment start: 2020-05-26T05:00:00+08:00.\n"
    "- Most severe segment: trace.self.jdbc.duration_p95 at 2020-05-26T05:22:00+08:00 (high), severity ≈ 5.15666; "
    "single-point spike (max=0.0505, P95≈0.008202).\n"
    "- Other notable high-severity short segments (length 1–2 minutes):\n"
    "  • trace.self.local.duration_p95 (low) at 05:17–05:18, severity ≈ 0.802.\n"
    "  • trace.from_os_021.remoteprocess.duration_p95 (low) at 05:17–05:18, severity ≈ 0.616.\n"
    "  • trace.self.csf.duration_mean (high) at 05:21–05:22, severity ≈ 0.520.\n"
    "  • trace.self.local.duration_mean (low) at 05:17–05:18, severity ≈ 0.517.\n"
    "- Most segments are short (1–2 minutes). The highest-severity event is a JDBC p95 spike at ~05:22.\n"
    "- Recommendation: investigate docker_003 around 05:16–05:22, focusing on the JDBC p95 spike and correlated local/remote process traces (especially interactions with os_021).\n"
    "- The full filtered table is stored as kernel variable: docker_003_trace_faults."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                      trace_name breach               start_ts_iso                 end_ts_iso  length  max_value_in_segment  min_value_in_segment          P95          P5  severity
212  docker_003                    trace.self.jdbc.duration_p95   high  2020-05-26T05:22:00+08:00  2020-05-26T05:22:00+08:00       1              0.050500              0.050500     0.008202    0.003000  5.156660
224  docker_003                   trace.self.local.duration_p95    low  2020-05-26T05:17:00+08:00  2020-05-26T05:18:00+08:00       2              0.008000              0.008000     0.098000    0.040400  0.801980
178  docker_003    trace.from_os_021.remoteprocess.duration_p95    low  2020-05-26T05:17:00+08:00  2020-05-26T05:18:00+08:00       2              0.627000              0.322150     2.519413    0.838613  0.615854
185  docker_003                    trace.self.csf.duration_mean   high  2020-05-26T05:21:00+08:00  2020-05-26T05:22:00+08:00       2              0.115947              0.078515     0.076275    0.034834  0.520111
220  docker_003                  trace.self.local.duration_mean    low  2020-05-26T05:17:00+08:00  2020-05-26T05:18:00+08:00       2              0.003684              0.003033     0.021527    0.006277  0.516834
208  docker_003                    trace.self.jdbc.duration_p95   high  2020-05-26T05:03:00+08:00  2020-05-26T05:03:00+08:00       1              0.011600              0.011600     0.008202    0.003000  0.414203
195  docker_003              trace.self.flyremote.duration_mean   high  2020-05-26T05:20:00+08:00  2020-05-26T05:20:00+08:00       1              0.017217              0.017217     0.012716    0.004942  0.353937
175  docker_003   trace.from_os_021.remoteprocess.duration_mean    low  2020-05-26T05:17:00+08:00  2020-05-26T05:18:00+08:00       2              0.237636              0.197188     0.894698    0.289823  0.319626
205  docker_003                   trace.self.jdbc.duration_mean    low  2020-05-26T05:17:00+08:00  2020-05-26T05:18:00+08:00       2              0.001316              0.001184     0.004669    0.001654  0.284333
187  docker_003                     trace.self.csf.duration_p95    low  2020-05-26T05:18:00+08:00  2020-05-26T05:19:00+08:00       2              0.052900              0.039000     0.195000    0.053958  0.277209
235  docker_003  trace.to_docker_005.remoteprocess.duration_p95   high  2020-05-26T05:20:00+08:00  2020-05-26T05:20:00+08:00       1              0.090000              0.090000     0.071182    0.036832  0.264356
179  docker_003    trace.from_os_021.remoteprocess.duration_p95   high  2020-05-26T05:21:00+08:00  2020-05-26T05:21:00+08:00       1              3.100800              3.100800     2.519413    0.838613  0.230763
176  docker_003   trace.from_os_021.remoteprocess.duration_mean   high  2020-05-26T05:21:00+08:00  2020-05-26T05:21:00+08:00       1              1.082220              1.082220     0.894698    0.289823  0.209592
184  docker_003                    trace.self.csf.duration_mean    low  2020-05-26T05:17:00+08:00  2020-05-26T05:18:00+08:00       2              0.034311              0.029023     0.076275    0.034834  0.166817
197  docker_003               trace.self.flyremote.duration_p95    low  2020-05-26T05:17:00+08:00  2020-05-26T05:17:00+08:00       1              0.005400              0.005400     0.091660    0.006293  0.141836
237  docker_003     trace.to_docker_005.remoteprocess.row_count    low  2020-05-26T05:18:00+08:00  2020-05-26T05:19:00+08:00       2            128.000000            112.000000   658.200000  129.800000  0.137134
190  docker_003                        trace.self.csf.row_count    low  2020-05-26T05:18:00+08:00  2020-05-26T05:19:00+08:00       2            128.000000            112.000000   658.200000  129.800000  0.137134
201  docker_003                  trace.self.flyremote.row_count    low  2020-05-26T05:18:00+08:00  2020-05-26T05:19:00+08:00       2             32.000000             28.000000   164.550000   32.450000  0.137134
181  docker_003       trace.from_os_021.remoteprocess.row_count    low  2020-05-26T05:18:00+08:00  2020-05-26T05:19:00+08:00       2             32.000000             28.000000   164.550000   32.450000  0.137134
217  docker_003                       trace.self.jdbc.row_count    low  2020-05-26T05:18:00+08:00  2020-05-26T05:19:00+08:00       2            512.000000            448.000000  2635.900000  519.200000  0.137134,        earliest_start_ts_iso        most_severe_trace_name   most_severe_start_ts_iso most_severe_breach  most_severe_severity  num_segments  num_unique_trace_names
0  2020-05-26T05:00:00+08:00  trace.self.jdbc.duration_p95  2020-05-26T05:22:00+08:00               high               5.15666            66                      24)```
```